In [1]:
# 1

import chess
import heapq
from typing import List


# Piece values for material evaluation
PIECE_VALUES = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}

# Large value for checkmate detection
CHECKMATE_SCORE = 10000


def evaluate_board(board: chess.Board) -> float:
    # Check for checkmate
    if board.is_checkmate():
        # If it's whites turn and checkmate, white lost
        if board.turn == chess.WHITE:
            return -CHECKMATE_SCORE
        else:
            return CHECKMATE_SCORE

    # Check for stalemate or draw
    if board.is_stalemate() or board.is_insufficient_material():
        return 0

    # Material counting
    score = 0
    for piece_type in PIECE_VALUES:
        # Count white pieces
        white_pieces = len(board.pieces(piece_type, chess.WHITE))
        # Count black pieces
        black_pieces = len(board.pieces(piece_type, chess.BLACK))
        # Add to score (white positive, black negative)
        score += PIECE_VALUES[piece_type] * (white_pieces - black_pieces)

    return score


def beam_search(board: chess.Board, beam_width: int, depth_limit: int):
    if depth_limit <= 0:
        return [], evaluate_board(board)

    # Determine if current player is white or black
    is_maximizing = board.turn == chess.WHITE

    # Initialize beam
    initial_eval = evaluate_board(board)
    beam = [(initial_eval, [], board.copy())]

    best_result = ([], initial_eval)

    for depth in range(depth_limit):
        candidates = []
        # Depth 0: original player's turn, Depth 1: opponent's turn, etc.
        current_maximizing = is_maximizing if depth % 2 == 0 else not is_maximizing

        for _, move_sequence, current_board in beam:
            legal_moves = list(current_board.legal_moves)

            # If no legal moves, this is either checkmate or stalemate
            if not legal_moves:
                eval_score = evaluate_board(current_board)
                candidates.append((eval_score, move_sequence, current_board.copy()))
                continue

            for move in legal_moves:
                # Make the move on a copy
                new_board = current_board.copy()
                new_board.push(move)

                # Evaluate the new position
                eval_score = evaluate_board(new_board)

                # Track the move sequence
                new_sequence = move_sequence + [move]

                candidates.append((eval_score, new_sequence, new_board))

        if not candidates:
            break

        # Select top beam_width candidates
        # For maximizing player: select highest scores (nlargest)
        # For minimizing player: select lowest scores (nsmallest)
        if current_maximizing:
            beam = heapq.nlargest(beam_width, candidates, key=lambda x: x[0])
        else:
            beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])

    # Return the best result from the final beam
    if beam:
        if is_maximizing:
            best = max(beam, key=lambda x: x[0])
        else:
            best = min(beam, key=lambda x: x[0])
        best_result = (best[1], best[0])

    return best_result


def format_move_sequence(moves: List[chess.Move], board: chess.Board):
    temp_board = board.copy()
    move_strs = []
    for move in moves:
        move_strs.append(temp_board.san(move))
        temp_board.push(move)
    return " -> ".join(move_strs)


# starting position
board = chess.Board()
beam_width = 4
depth_limit = 5

print("Starting position\n")
print(f"Board:\n{board}\n")
print(f"Beam Width: {beam_width}")
print(f"Depth Limit: {depth_limit}")
print(f"Turn: {'White' if board.turn == chess.WHITE else 'Black'}")
print()

best_moves, score = beam_search(board, beam_width, depth_limit)

print(f"Best Move Sequence: {format_move_sequence(best_moves, board)}")
print(f"Evaluation Score: {score}")
print()

# Position with checkmate
tactical_fen = "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR w KQkq - 4 4"
board2 = chess.Board(tactical_fen)

print("Checkmate Opportunity\n")
print(f"Board:\n{board2}\n")
print(f"FEN: {tactical_fen}")
print(f"Beam Width: {beam_width}")
print(f"Depth Limit: {depth_limit}")
print()

best_moves2, score2 = beam_search(board2, beam_width, depth_limit)

print(f"Best Move Sequence: {format_move_sequence(best_moves2, board2)}")
print(f"Evaluation Score: {score2}")
print()

# Endgame position
endgame_fen = "8/8/8/5k2/8/8/4Q3/4K3 w - - 0 1"
board3 = chess.Board(endgame_fen)

print("Endgame position\n")
print(f"Board:\n{board3}\n")
print(f"FEN: {endgame_fen}")
print(f"Beam Width: {beam_width}")
print(f"Depth Limit: {depth_limit}")
print()

best_moves3, score3 = beam_search(board3, beam_width, depth_limit)

print(f"Best Move Sequence: {format_move_sequence(best_moves3, board3)}")
print(f"Evaluation Score: {score3}")


Starting position

Board:
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R

Beam Width: 4
Depth Limit: 5
Turn: White

Best Move Sequence: Nh3 -> Nh6 -> Ng5 -> Rg8 -> Nxh7
Evaluation Score: 1

Checkmate Opportunity

Board:
r . b q k b n r
p p p p . p p p
. . n . . . . .
. . . . p . . .
. . B . P . . .
. . . . . Q . .
P P P P . P P P
R N B . K . N R

FEN: r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR w KQkq - 4 4
Beam Width: 4
Depth Limit: 5

Best Move Sequence: Be6 -> Nge7 -> Bxd7+ -> Kxd7 -> Qxf7
Evaluation Score: -1

Endgame position

Board:
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . k . .
. . . . . . . .
. . . . . . . .
. . . . Q . . .
. . . . K . . .

FEN: 8/8/8/5k2/8/8/4Q3/4K3 w - - 0 1
Beam Width: 4
Depth Limit: 5

Best Move Sequence: Qe8 -> Kf6 -> Qh8+ -> Kf7 -> Qg8+
Evaluation Score: 9


In [2]:
# 2

import math
from typing import List, Sequence, Tuple

# Coordinate type: accepts both int and float values
Coord = Tuple[int | float, int | float]


def euclidean_distance(p1: Coord, p2: Coord) -> float:
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def total_route_distance(route: List[int], coords: Sequence[Coord]) -> float:
    distance = 0.0
    n = len(route)
    for i in range(n):
        distance += euclidean_distance(coords[route[i]], coords[route[(i + 1) % n]])
    return distance


def hill_climbing(coords: Sequence[Coord]) -> Tuple[List[int], float]:
    n = len(coords)
    if n == 0:
        return [], 0.0
    if n == 1:
        return [0], 0.0

    current_route = list(range(n))
    current_distance = total_route_distance(current_route, coords)

    print(f"Initial route:    {current_route}")
    print(f"Initial distance: {current_distance:.4f}\n")

    improved = True
    iteration = 0

    while improved:
        improved = False
        best_route = current_route[:]
        best_distance = current_distance

        for i in range(1, n - 1):
            for j in range(i + 1, n):
                neighbour = (
                    current_route[:i]
                    + current_route[i : j + 1][::-1]
                    + current_route[j + 1 :]
                )
                neighbour_distance = total_route_distance(neighbour, coords)

                # Accept only if this neighbour is strictly better (mirrors
                # the reference: move only when heuristic improves)
                if neighbour_distance < best_distance:
                    best_route = neighbour
                    best_distance = neighbour_distance
                    improved = True

        # Move to the best improving neighbour found
        if improved:
            iteration += 1
            print(
                f"Iteration {iteration}: distance improved {current_distance:.4f} -> {best_distance:.4f}  route: {best_route}"
            )
            current_route = best_route
            current_distance = best_distance

    if not improved and iteration == 0:
        print("Already at local optimum. No improvement found.")

    return current_route, current_distance


# 1: Small delivery grid
print("1: 5 Delivery Locations (Grid)")

locations_1 = [
    (0, 0),  # Depot  (index 0)
    (2, 4),  # Stop 1 (index 1)
    (5, 2),  # Stop 2 (index 2)
    (6, 6),  # Stop 3 (index 3)
    (1, 7),  # Stop 4 (index 4)
]

optimized_route, optimized_distance = hill_climbing(locations_1)

print(f"\nOptimized route (indices): {optimized_route}")
print(f"Optimized route (coords):  {[locations_1[i] for i in optimized_route]}")
print(f"Total distance:            {optimized_distance:.4f}")

# 2: Larger to see more swaps
print()
print("2: 8 Delivery Locations")

locations_2 = [
    (0, 0),  # Depot
    (3, 1),
    (6, 4),
    (5, 8),
    (2, 9),
    (0, 5),
    (4, 6),
    (8, 2),
]

optimized_route_2, optimized_distance_2 = hill_climbing(locations_2)

print(f"\nOptimized route (indices): {optimized_route_2}")
print(f"Optimized route (coords):  {[locations_2[i] for i in optimized_route_2]}")
print(f"Total distance:            {optimized_distance_2:.4f}")


1: 5 Delivery Locations (Grid)
Initial route:    [0, 1, 2, 3, 4]
Initial distance: 24.3709

Iteration 1: distance improved 24.3709 -> 22.2417  route: [0, 1, 4, 3, 2]
Iteration 2: distance improved 22.2417 -> 22.2417  route: [0, 2, 3, 4, 1]

Optimized route (indices): [0, 2, 3, 4, 1]
Optimized route (coords):  [(0, 0), (5, 2), (6, 6), (1, 7), (2, 4)]
Total distance:            22.2417

2: 8 Delivery Locations
Initial route:    [0, 1, 2, 3, 4, 5, 6, 7]
Initial distance: 37.1886

Iteration 1: distance improved 37.1886 -> 35.2166  route: [0, 1, 6, 5, 4, 3, 2, 7]
Iteration 2: distance improved 35.2166 -> 33.5041  route: [0, 1, 6, 7, 2, 3, 4, 5]
Iteration 3: distance improved 33.5041 -> 30.7607  route: [0, 1, 2, 7, 6, 3, 4, 5]
Iteration 4: distance improved 30.7607 -> 28.7886  route: [0, 1, 7, 2, 6, 3, 4, 5]

Optimized route (indices): [0, 1, 7, 2, 6, 3, 4, 5]
Optimized route (coords):  [(0, 0), (3, 1), (8, 2), (6, 4), (4, 6), (5, 8), (2, 9), (0, 5)]
Total distance:            28.7886


In [3]:
# 3

import numpy as np
import random

# Generate 10 random cities (x, y)
num_cities = 10
cities = np.random.rand(num_cities, 2) * 100


def calculate_distance(order):
    # Calculates total distance of a specific route
    dist = 0
    for i in range(len(order)):
        city_a = cities[order[i]]
        city_b = cities[order[(i + 1) % len(order)]]  # Loop back to start
        dist += np.linalg.norm(city_a - city_b)
    return dist


# GA Components
def initial_population(pop_size, num_cities):
    return [random.sample(range(num_cities), num_cities) for _ in range(pop_size)]


def crossover(parent1, parent2):
    # Ordered Crossover to maintain valid permutations
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))

    child = [None] * size
    child[start:end] = parent1[start:end]

    # Fill remaining slots with parent2 genes in order
    p2_remaining = [item for item in parent2 if item not in child]
    for i in range(size):
        if child[i] is None:
            child[i] = p2_remaining.pop(0)
    return child


def mutate(individual, mutation_rate=0.05):
    # Swap mutation: swap two cities in the route.
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(individual)), 2)
        individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual


# The Evolution Loop
def evolve_tsp(pop_size=100, generations=500):
    pop = initial_population(pop_size, num_cities)

    for _ in range(generations):
        # Sort population by fitness (shorter distance = better)
        pop = sorted(pop, key=lambda x: calculate_distance(x))

        # Keep the top 10%
        new_gen = pop[: pop_size // 10]

        # Fill the rest with offspring
        while len(new_gen) < pop_size:
            p1, p2 = random.sample(pop[:50], 2)  # Select from top 50
            child = crossover(p1, p2)
            new_gen.append(mutate(child))

        pop = new_gen

    best_route = pop[0]
    return best_route, calculate_distance(best_route)


best_route, best_dist = evolve_tsp()
print(f"Best Route: {best_route}")
print(f"Distance: {best_dist:.2f}")


Best Route: [6, 7, 2, 4, 8, 0, 1, 3, 5, 9]
Distance: 334.33


In [4]:
# 4

def simple_beam_search(tasks, num_processors, beam_width=2):
    # Priority first (high to low), then time (long to short)
    tasks.sort(key=lambda x: (x["priority"], x["time"]), reverse=True)

    beam = [[0] * num_processors]

    for task in tasks:
        all_possibilities = []

        for current_loads in beam:
            for i in range(num_processors):
                next_loads = list(current_loads)  # copy current loads
                next_loads[i] += task["time"]  # add task to processor i
                all_possibilities.append(next_loads)

        # Keep only the top 'beam_width' results for the next task
        all_possibilities.sort(key=lambda x: max(x))
        beam = all_possibilities[:beam_width]

    best_load_distribution = beam[0]
    return best_load_distribution


# Example Data
my_tasks = [
    {"id": "T1", "time": 10, "priority": 3},
    {"id": "T2", "time": 20, "priority": 1},
    {"id": "T3", "time": 15, "priority": 3},
]

final_loads = simple_beam_search(my_tasks, num_processors=2)
print(f"Final loads on processors: {final_loads}")
print(f"The maximum load is: {max(final_loads)}")


Final loads on processors: [15, 30]
The maximum load is: 30
